# Layerwise Inference Experiment (MNIST) - Step-by-step

In [1]:
import sys
from pathlib import Path

# Notebook is in scripts/, project root is one level up
ROOT = Path("..").resolve()

# Make "import src...." work
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Checkpoint exists?", (ROOT / "fcn_mnist_best.pt").exists())

Project root: C:\Users\tulin\Documents\DTU\Bachelor Project
Checkpoint exists? True


In [2]:
import torch
import torch.nn as nn

from src.exp_utils import (
    get_device,
    load_test_loader,
    build_model,
    load_weights,
    estimate_fp32_weight_bytes,
    estimate_peak_decompressed_layer_bytes,
    fmt_bytes,
    save_compressed_model,
    load_compressed_model,
    estimate_compressed_storage_bytes_from_file,
)
from src.training import evaluate
from src.pruning import magnitude_prune_linear_layers, make_pruning_permanent, model_sparsity
from src.layerwise_inference import layerwise_evaluate_accuracy

## Setup

In [3]:
device = get_device()
loss_fn = nn.CrossEntropyLoss()
test_loader = load_test_loader()

ckpt_path = ROOT / "fcn_mnist_best.pt"
prune_amount = 0.5  # choose based on sweep

print("Device:", device)
print("Checkpoint:", ckpt_path)
print("Prune amount:", prune_amount)

100.0%
100.0%
100.0%
100.0%

Device: cpu
Checkpoint: C:\Users\tulin\Documents\DTU\Bachelor Project\fcn_mnist_best.pt
Prune amount: 0.5


## Baseline FP32 Accuracy
We load the trained checkpoint into a fresh FCN and evaluate it normally.

In [4]:
base_model = build_model(device)
load_weights(base_model, str(ckpt_path), device)

base_loss, base_accuracy = evaluate(base_model, test_loader, loss_fn, device)

print(f"Baseline FP32 accuracy: {base_accuracy:.4f} (loss {base_loss:.4f})")

Baseline FP32 accuracy: 0.9750 (loss 0.0786)


In [5]:
print(base_model)

FCN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (net): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


# Pruned FP32 accuracy
We prune the model, make pruning permanent (real zeros), then evaluate again.

In [6]:
pruned_model = build_model(device)
load_weights(pruned_model, str(ckpt_path), device)

if prune_amount > 0.0:
    magnitude_prune_linear_layers(pruned_model, amount=prune_amount)
    make_pruning_permanent(pruned_model)

pruned_loss, pruned_accuracy = evaluate(pruned_model, test_loader, loss_fn, device)
sparsity = model_sparsity(pruned_model) * 100.0

print(f"Pruned FP32 accuracy: {pruned_accuracy:.4f} (loss {pruned_loss:.4f})")
print(f"Sparsity: {sparsity:.2f}%")

Pruned FP32 accuracy: 0.9738 (loss 0.0879)
Sparsity: 50.00%


# Export compressed model (int8 weights + scale + bias)
We save the pruned model in a layer-by-layer compressed format.

In [7]:
save_path = ROOT / f"fcn_mnist_pruned_{int(prune_amount*100)}_int8_compressed.pt"
save_compressed_model(pruned_model, str(save_path))

print("Saved compressed model:", save_path)
print("Compressed file exists?", save_path.exists())

Saved compressed model: C:\Users\tulin\Documents\DTU\Bachelor Project\fcn_mnist_pruned_50_int8_compressed.pt
Compressed file exists? True


# Load compressed model and inspect its structure
This is the part that makes it easier to understand what is stored.

In [8]:
compressed_loaded = load_compressed_model(str(save_path))

print("Loaded compressed model type:", type(compressed_loaded))

if isinstance(compressed_loaded, dict):
    print("Top-level keys:", list(compressed_loaded.keys()))

Loaded compressed model type: <class 'dict'>
Top-level keys: ['format_version', 'model_type', 'in_dim', 'layers']


In [9]:
layers = None

if isinstance(compressed_loaded, dict):
    for candidate in ["layers", "packed_layers", "linear_layers"]:
        if candidate in compressed_loaded and isinstance(compressed_loaded[candidate], list):
            layers = compressed_loaded[candidate]
            print("Found layer list under key:", candidate)
            break

if layers is None:
    print("Could not auto-detect per-layer list. If you know the key, set:")
    print("layers = compressed_loaded['YOUR_KEY']")
else:
    print("Number of layers:", len(layers))
    print("Keys in layer 0:", list(layers[0].keys()))

Found layer list under key: layers
Number of layers: 5
Keys in layer 0: ['type', 'in_features', 'out_features', 'W_q', 's_w', 'b']


# Layerwise inference accuracy (CPU)
We run layerwise inference where only one layer is decompressed at a time.

In [10]:
layer_device = torch.device("cpu")
lw_accuracy = layerwise_evaluate_accuracy(compressed_loaded, test_loader, layer_device)

print(f"Layerwise int8 accuracy: {lw_accuracy:.4f}")

Layerwise int8 accuracy: 0.9737


# Storage and peak decompressed layer estimate
We estimate:
- FP32 weight storage
- compressed storage on disk
- largest layer in FP32 (peak temporary RAM for one decompressed layer)

In [11]:
fp32_weight_bytes = estimate_fp32_weight_bytes(pruned_model)
compressed_bytes = estimate_compressed_storage_bytes_from_file(str(save_path))
peak_layer_fp32_bytes = estimate_peak_decompressed_layer_bytes(pruned_model)

ratio = (fp32_weight_bytes / compressed_bytes) if compressed_bytes > 0 else float("inf")

print("\n[Storage (weights/scales/bias)]")
print("FP32 weights:           ", fmt_bytes(fp32_weight_bytes))
print("Compressed (int8+meta): ", fmt_bytes(compressed_bytes))
print("Compression ratio:      ", f"{ratio:.2f}x")

print("\n[Peak decompressed weights]")
print("Largest layer (FP32):   ", fmt_bytes(peak_layer_fp32_bytes))


[Storage (weights/scales/bias)]
FP32 weights:            2,140,160 B (2.04 MB)
Compressed (int8+meta):  538,164 B (525.55 KB)
Compression ratio:       3.98x

[Peak decompressed weights]
Largest layer (FP32):    1,605,632 B (1.53 MB)


# Final Summary

In [12]:
drop = base_accuracy - lw_accuracy

print("\n" + "=" * 70)
print("Layerwise Inference Experiment (MNIST)".center(70))
print("=" * 70)

print("\n[Setup]")
print(f"  Checkpoint     : {ckpt_path}")
print(f"  Prune amount   : {prune_amount:.2f}")
print(f"  Sparsity       : {sparsity:.2f}%")

print("\n[Accuracy]")
print(f"  Baseline FP32  : {base_accuracy:.4f}   (loss {base_loss:.4f})")
print(f"  Pruned FP32    : {pruned_accuracy:.4f}   (loss {pruned_loss:.4f})")
print(f"  Layerwise int8 : {lw_accuracy:.4f}")
print(f"  Drop vs base   : {drop:.4f}")

print("\n[Storage (weights/scales/bias)]")
print(f"  FP32 weights           : {fmt_bytes(fp32_weight_bytes)}")
print(f"  Compressed (int8+meta) : {fmt_bytes(compressed_bytes)}")
print(f"  Compression ratio      : {ratio:.2f}x")

print("\n[Peak decompressed weights]")
print(f"  Largest layer (FP32)   : {fmt_bytes(peak_layer_fp32_bytes)}")
print("  (Temporary RAM used for the decompressed layer during inference.)")

print("\n[Artifacts]")
print(f"  Saved compressed model : {save_path}")
print("=" * 70 + "\n")


                Layerwise Inference Experiment (MNIST)                

[Setup]
  Checkpoint     : C:\Users\tulin\Documents\DTU\Bachelor Project\fcn_mnist_best.pt
  Prune amount   : 0.50
  Sparsity       : 50.00%

[Accuracy]
  Baseline FP32  : 0.9750   (loss 0.0786)
  Pruned FP32    : 0.9738   (loss 0.0879)
  Layerwise int8 : 0.9737
  Drop vs base   : 0.0013

[Storage (weights/scales/bias)]
  FP32 weights           : 2,140,160 B (2.04 MB)
  Compressed (int8+meta) : 538,164 B (525.55 KB)
  Compression ratio      : 3.98x

[Peak decompressed weights]
  Largest layer (FP32)   : 1,605,632 B (1.53 MB)
  (Temporary RAM used for the decompressed layer during inference.)

[Artifacts]
  Saved compressed model : C:\Users\tulin\Documents\DTU\Bachelor Project\fcn_mnist_pruned_50_int8_compressed.pt

